<a href="https://colab.research.google.com/github/rhyan-rpone/rponeconsultoria/blob/main/Aura_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
# Instalando as libs
!pip install -U langgraph langchain langchain_community langchain_groq pydantic
!pip install -U langchain-openai
# Para visualização do grafo
!apt-get install -y graphviz libgraphviz-dev
!pip install pygraphviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.8 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
graphviz is already the newest version (2.42.2-6ubuntu0.1).
libgraphviz-dev is already the newest version (2.42.2-6ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [48]:
#  IMPORTANDO E DEFININDO MODELO/LLM

import os
from google.colab import userdata
from langchain_openai import ChatOpenAI

# 1. Autenticação com a chave da OpenAI
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# 2. Configurando o GPT-4o-mini (Rápido, muito barato e excelente para lógica)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0  # Mantemos 0 para o agente ser analítico e não alucinar dados
)

# Teste rápido
try:
    resposta = llm.invoke("Responda apenas com 'OK' se estiver funcionando.")
    print(f"✅ Motor OpenAI conectado! Resposta: {resposta.content}")
except Exception as e:
    print(f"❌ Erro de conexão: {e}")

✅ Motor OpenAI conectado! Resposta: OK


In [75]:
# CRIANDO LEAD PROFILE (PERFIL DO CLIENTE - DATA SCHEMA)

from pydantic import BaseModel, Field
from typing import Optional, List

class LeadProfile(BaseModel):
    budget_max: Optional[float] = Field(None, description="Orçamento máximo do cliente em reais")
    location_pref: Optional[str] = Field(None, description="Cidade ou bairro de preferência")
    property_type: Optional[str] = Field(None, description="Tipo: Casa, Apto, Terreno, etc")
    must_haves: List[str] = Field(default=[], description="Itens indispensáveis (ex: piscina, 3 quartos)")
    urgency: Optional[str] = Field(None, description="Nível de urgência: Baixa, Média, Alta")

In [64]:
# DEFINIÇÃO DO ESTADO DO AGENTE
# O Estado é o "coração" do LangGraph. Tudo o que o agente aprender ou
# decidir será guardado aqui dentro para que os próximos nós possam ler.

from typing import TypedDict, Annotated, List
import operator

class AgentState(TypedDict):
    # 'Annotated' com 'operator.add' permite que o LangGraph anexe novas mensagens ao histórico em vez de apagar as antigas, garantindo a memória do chat.
    messages: Annotated[list, operator.add]

    # Dicionário para guardar o perfil extraído (Passo 3)
    user_data: dict

    # Lista para guardar os imóveis encontrados (Passo 5)
    potential_houses: List[str]

    # O System Prompt agora é muito mais focado na conversa
instrucoes = """Você é a Aura, consultora da Aura Imobiliária.
Sua missão é conversar com o cliente de forma natural.

FLUXO DE TRABALHO:
1. Comece sempre com uma saudação amigável.
2. À medida que o cliente fala o que quer, use a ferramenta 'salvar_perfil_cliente'.
3. Quando tiver informações de local e preço (ou se o cliente pedir), use 'buscar_imoveis_disponiveis'.
4. Nunca responda com JSON ou códigos. Responda como uma pessoa real e elegante.
5. Se não encontrar imóveis, sugira mudar o filtro de preço ou cidade."""

In [69]:
# DEFININDO AS TOOLS: Definindo as ferramentas disponíveis pro nosso agente Aura utilizar

from langchain_core.tools import tool

@tool
def salvar_perfil_cliente(budget_max: float = None, location_pref: str = None, property_type: str = None):
    """Utilize esta ferramenta sempre que o cliente mencionar valores, cidades ou tipos de imóvel.
    Isso salva as preferências no banco de dados para filtrar a busca."""
    # Essa função agora apenas sinaliza que os dados foram captados
    return "Perfil atualizado com sucesso."

@tool
def buscar_imoveis_disponiveis(budget: float = 99999999, localizacao: str = ""):
    """Utilize esta ferramenta para consultar o inventário de imóveis.
    Use-a quando o cliente pedir opções ou quando você já tiver dados suficientes para sugerir algo."""
    mock_db = [
        {"id": 1, "nome": "Apto Vista Alegre", "preco": 550000, "cidade": "Paulínia"},
        {"id": 2, "nome": "Casa Garden", "preco": 850000, "cidade": "Paulínia"},
        {"id": 3, "nome": "Studio Central", "preco": 300000, "cidade": "Campinas"},
        {"id": 4, "nome": "Mansão das Palmeiras", "preco": 2500000, "cidade": "Paulínia"},
        # ... (os outros imóveis)
    ]

    localizacao = localizacao.lower()
    matches = [f"{i['nome']} (R$ {i['preco']:,} em {i['cidade']})" for i in mock_db
               if i["preco"] <= budget and (localizacao in i["cidade"].lower() or localizacao == "")]

    return matches if matches else "Nenhum imóvel encontrado para este perfil."

tools = [salvar_perfil_cliente, buscar_imoveis_disponiveis]

In [76]:
# 1. Ajuste no Import (conforme o aviso do seu console)
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

# 2. Definimos o motor
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3. Instruções do Sistema
instrucoes = """Você é a Aura, consultora da Aura Imobiliária.
Sua missão é conversar com o cliente de forma natural e elegante.

FLUXO DE TRABALHO:
- Comece com uma saudação amigável.
- Use 'salvar_perfil_cliente' quando o cliente mencionar preferências (cidade, preço, tipo).
- Use 'buscar_imoveis_disponiveis' quando o cliente pedir opções ou você já tiver dados para sugerir.
- Mantenha o tom profissional e prestativo. Nunca mostre códigos ou JSON para o usuário."""

# 4. Criamos a memória
memory = MemorySaver()

# 5. Criamos o agente (Usando o parâmetro 'prompt' em vez de 'state_modifier')
app = create_react_agent(
    llm,
    tools=tools,
    checkpointer=memory,
    prompt=instrucoes  # <--- Mudança crucial aqui
)

print("✅ Agente Aura reconstruído com sucesso e sem avisos de depreciação!")

✅ Agente Aura reconstruído com sucesso e sem avisos de depreciação!


/tmp/ipykernel_26976/1619299077.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  app = create_react_agent(


In [77]:
# CRIANDO BANCO DE DADOS MOCK, LOGICA DE RECUPERAÇÃO DE PERFIL E LOGICA DE FILTRO PRA QUERY:

def search_node(state: AgentState):
    print("--- [LOG] BUSCANDO NO INVENTÁRIO---")

    # Recuperamos o perfil que o extractor acabou de criar
    profile = state.get("user_data", {})
    budget = profile.get("budget_max") or 999999999
    loc = (profile.get("location_pref") or "").lower()

    # Banco de Dados Mockado com 10 opções variadas (Poderia ser qualquer conexão de DB como SUpabase)
    mock_db = [
        {"id": 1, "nome": "Apto Jd. Vista Alegre", "preco": 550000, "cidade": "Paulínia"},
        {"id": 2, "nome": "Casa Garden", "preco": 850000, "cidade": "Paulínia"},
        {"id": 3, "nome": "Studio Central", "preco": 300000, "cidade": "Campinas"},
        {"id": 4, "nome": "Mansão das Palmeiras", "preco": 2500000, "cidade": "Paulínia"},
        {"id": 5, "nome": "Cobertura Sky View", "preco": 1800000, "cidade": "Campinas"},
        {"id": 6, "nome": "Terreno Alphaville", "preco": 950000, "cidade": "Campinas"},
        {"id": 7, "nome": "Casa de Vila Tranquila", "preco": 420000, "cidade": "Paulínia"},
        {"id": 8, "nome": "Apto Premium Nova Veneza", "preco": 680000, "cidade": "Paulínia"},
        {"id": 9, "nome": "Loft Industrial Cambuí", "preco": 720000, "cidade": "Campinas"},
        {"id": 10, "nome": "Chácara Recanto Real", "preco": 4500000, "cidade": "Paulínia"}
    ]

    # Lógica de filtro refinada
    # Filtra por preço (menor ou igual ao budget) e por cidade (se informada)
    matches = [
        f"{imovel['nome']} (R$ {imovel['preco']:,} em {imovel['cidade']})"
        for imovel in mock_db
        if imovel["preco"] <= budget and (loc in imovel["cidade"].lower() or loc == "")
    ]

    # Atualizamos o estado com a lista de imóveis que passaram no filtro
    return {"potential_houses": matches}

In [78]:
# 1. Definimos o modelo (LLM)
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. Criamos a memória (Equivalente ao seu passo 4 antigo)
memory = MemorySaver()

# 3. COMPILAÇÃO DO AGENTE
# O create_react_agent cria o grafo automaticamente baseado nas ferramentas
app = create_react_agent(
    llm,
    tools=tools,
    checkpointer=memory,
    prompt=instrucoes
)

print("Agente Aura compilado e pronto!")

✅ Agente Reativo Aura compilado e pronto!


/tmp/ipykernel_26976/3806628510.py:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  app = create_react_agent(


In [ ]:
def iniciar_chat_aura():
    config = {"configurable": {"thread_id": "usuario_real_01"}}
    print("🏠 Aura: Olá! Sou a Aura, sua consultora imobiliária. Como posso te ajudar hoje?\n")

    while True:
        user_input = input("👤 Você: ")
        if user_input.lower() in ["sair", "exit"]: break

        inputs = {"messages": [("user", user_input)]}

        # O agente decide se fala ou se usa ferramenta
        for s in app.stream(inputs, config, stream_mode="values"):
            message = s["messages"][-1]

        # Imprime apenas a resposta final do Agente para o usuário
        if message.content:
            print(f"\n✨ Aura: {message.content}\n" + "-"*30)

iniciar_chat_aura()

🏠 Aura: Olá! Sou a Aura, sua consultora imobiliária. Como posso te ajudar hoje?

👤 Você: oi, tudo bem?

✨ Aura: Olá! Tudo ótimo, e você? Como posso ajudá-lo hoje?
------------------------------
👤 Você: Gostaria de ver uns imoveis

✨ Aura: Claro! Para que eu possa ajudá-lo da melhor forma, você poderia me informar em qual cidade está procurando imóveis e qual é o seu orçamento? Além disso, se tiver um tipo específico de imóvel em mente, como apartamento ou casa, isso também seria útil.
------------------------------
👤 Você: Paulínia, imoveis de ate 500 mil reais, tenho preferencia por atamento com pelo menos 2 quartos

✨ Aura: Atualizei suas preferências com sucesso! No entanto, infelizmente, não encontrei imóveis disponíveis em Paulínia que atendam aos seus critérios de até 500 mil reais e com pelo menos 2 quartos.

Se desejar, posso ajudar a ajustar suas preferências ou buscar opções em cidades próximas. O que você acha?
------------------------------
👤 Você: busque outras opções

✨ A